# NEPA Diffusion Decoder - CIFAR-10 Training

This notebook trains a UNet diffusion decoder conditioned on NEPA embeddings for image reconstruction.

**Workflow:**
1. Setup & Dependencies
2. Load Pretrained NEPA Encoder
3. Create Diffusion Decoder
4. Train Diffusion Model
5. Inference & Reconstruction
6. Visualize Results
7. Save/Load Checkpoints

**Requirements:**
- GPU runtime (T4 or better)
- Pretrained NEPA encoder (train with `nepa_cifar10_colab.ipynb` first)

## 1. Setup & Dependencies

In [ ]:
# Check GPU availability
!nvidia-smi

In [ ]:
# Clone the NEPA repository
!git clone https://github.com/mu-hashmi/nepa.git
%cd nepa
!git checkout extended

In [ ]:
# Install dependencies
!pip install -q transformers==4.56.2 datasets==3.6.0 accelerate>=0.12.0
!pip install -q diffusers>=0.20.0
!pip install -q evaluate scikit-learn timm
!pip install -q wandb  # Optional: for experiment tracking

In [ ]:
# Import required libraries
import os
import sys
import torch
import torch.nn.functional as F
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from datasets import load_dataset
from torchvision.transforms import Compose, Resize, CenterCrop, ToTensor, Normalize, RandomHorizontalFlip, RandomResizedCrop
from diffusers import DDPMScheduler, DDIMScheduler

# Add models to path
sys.path.insert(0, '.')

from models.vit_nepa import ViTNepaModel, ViTNepaConfig
from models.diffusion_decoder import DiffusionDecoderConfig, NepaConditionedDiffusionForReconstruction

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Load Pretrained NEPA Encoder

**Important:** You need a pretrained NEPA encoder. Either:
- Train it first using `nepa_cifar10_colab.ipynb`
- Or load from Google Drive if you saved one previously

In [ ]:
# Option 1: Train NEPA encoder first (if not already done)
# Uncomment and run this cell if you need to train NEPA first

# !python run_nepa.py \
#     --config_name configs/pretrain/nepa-tiny-patch14-224-cifar10 \
#     --image_processor_name configs/pretrain/nepa-tiny-patch14-224-cifar10 \
#     --dataset_name cifar10 \
#     --load_from_disk False \
#     --do_train \
#     --output_dir outputs/nepa-tiny-patch14-224-cifar10 \
#     --num_train_epochs 100 \
#     --per_device_train_batch_size 128 \
#     --learning_rate 2e-4 \
#     --lr_scheduler_type cosine \
#     --warmup_ratio 0.05 \
#     --weight_decay 0.05 \
#     --logging_steps 50 \
#     --save_steps 5000 \
#     --seed 1337 \
#     --bf16 True \
#     --dataloader_num_workers 2 \
#     --remove_unused_columns False \
#     --report_to none

In [ ]:
# Option 2: Load from Google Drive (if you saved a checkpoint)
# Mount Drive first
from google.colab import drive
drive.mount('/content/drive')

# Copy checkpoint from Drive if it exists
import shutil
drive_nepa_path = "/content/drive/MyDrive/nepa-checkpoints/nepa-tiny-cifar10-pretrain"
local_nepa_path = "outputs/nepa-tiny-patch14-224-cifar10"

if os.path.exists(drive_nepa_path):
    os.makedirs(local_nepa_path, exist_ok=True)
    shutil.copytree(drive_nepa_path, local_nepa_path, dirs_exist_ok=True)
    print(f"Loaded NEPA checkpoint from Drive")
else:
    print(f"No checkpoint found at {drive_nepa_path}")
    print("Please train NEPA first using Option 1 above")

In [ ]:
# Verify NEPA encoder loads correctly
NEPA_MODEL_PATH = "outputs/nepa-tiny-patch14-224-cifar10"

nepa_encoder = ViTNepaModel.from_pretrained(NEPA_MODEL_PATH)
nepa_encoder.eval()

print(f"NEPA encoder loaded!")
print(f"Hidden size: {nepa_encoder.config.hidden_size}")
print(f"Num layers: {nepa_encoder.config.num_hidden_layers}")
print(f"Parameters: {sum(p.numel() for p in nepa_encoder.parameters()):,}")

## 3. Create Diffusion Decoder

In [ ]:
# Diffusion decoder configuration
DIFFUSION_CONFIG = DiffusionDecoderConfig(
    nepa_model_path=NEPA_MODEL_PATH,
    nepa_hidden_size=384,  # Must match NEPA encoder
    nepa_num_patches=256,  # 16x16 patches
    image_size=224,
    in_channels=3,
    out_channels=3,
    block_out_channels=(64, 128, 256, 512),
    layers_per_block=2,
    attention_head_dim=8,
    cross_attention_dim=384,
    num_train_timesteps=1000,
    num_inference_timesteps=50,
    beta_schedule="scaled_linear",
    prediction_type="epsilon",
    freeze_nepa=True,
)

print("Diffusion config created!")
print(f"UNet channels: {DIFFUSION_CONFIG.block_out_channels}")
print(f"Training timesteps: {DIFFUSION_CONFIG.num_train_timesteps}")
print(f"Inference timesteps: {DIFFUSION_CONFIG.num_inference_timesteps}")

In [ ]:
# Create diffusion model
model = NepaConditionedDiffusionForReconstruction(DIFFUSION_CONFIG)
model.load_nepa_encoder(NEPA_MODEL_PATH)

# Move to GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())

print(f"Model created!")
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Frozen parameters: {total_params - trainable_params:,}")

## 4. Train Diffusion Model

In [ ]:
# Load CIFAR-10 dataset
dataset = load_dataset("cifar10")

print(f"Train samples: {len(dataset['train'])}")
print(f"Test samples: {len(dataset['test'])}")

In [ ]:
# Data transforms - normalize to [-1, 1] for diffusion
train_transforms = Compose([
    RandomResizedCrop(224, scale=(0.8, 1.0), interpolation=Image.BICUBIC),
    RandomHorizontalFlip(),
    ToTensor(),
    Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
])

val_transforms = Compose([
    Resize(256, interpolation=Image.BICUBIC),
    CenterCrop(224),
    ToTensor(),
    Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
])

def preprocess_train(examples):
    examples["pixel_values"] = [train_transforms(img.convert("RGB")) for img in examples["img"]]
    return examples

def preprocess_val(examples):
    examples["pixel_values"] = [val_transforms(img.convert("RGB")) for img in examples["img"]]
    return examples

# Apply transforms
train_dataset = dataset["train"]
train_dataset.set_transform(preprocess_train)

val_dataset = dataset["test"]
val_dataset.set_transform(preprocess_val)

print("Transforms applied!")

In [ ]:
# Training hyperparameters
TRAIN_CONFIG = {
    "batch_size": 32,
    "learning_rate": 1e-4,
    "num_epochs": 100,
    "warmup_steps": 500,
    "log_interval": 100,
    "save_interval": 10,  # epochs
    "eval_interval": 5,   # epochs
}

print("Training config:")
for k, v in TRAIN_CONFIG.items():
    print(f"  {k}: {v}")

In [ ]:
from torch.utils.data import DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from tqdm.auto import tqdm

# Data loaders
def collate_fn(examples):
    pixel_values = torch.stack([ex["pixel_values"] for ex in examples])
    return {"pixel_values": pixel_values}

train_loader = DataLoader(
    train_dataset,
    batch_size=TRAIN_CONFIG["batch_size"],
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=2,
    pin_memory=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=TRAIN_CONFIG["batch_size"] * 2,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=2,
)

# Optimizer and scheduler
optimizer = AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=TRAIN_CONFIG["learning_rate"],
    weight_decay=0.01,
)

total_steps = len(train_loader) * TRAIN_CONFIG["num_epochs"]
scheduler = CosineAnnealingLR(optimizer, T_max=total_steps)

print(f"Total training steps: {total_steps}")
print(f"Steps per epoch: {len(train_loader)}")

In [ ]:
# Training loop
import copy

# EMA model for decoder
ema_decay = 0.9999
ema_decoder = copy.deepcopy(model.decoder)
ema_decoder.eval()
for p in ema_decoder.parameters():
    p.requires_grad_(False)

def update_ema(ema_model, model, decay):
    with torch.no_grad():
        for ema_p, p in zip(ema_model.parameters(), model.parameters()):
            ema_p.mul_(decay).add_(p, alpha=1 - decay)

# Training
model.train()
global_step = 0
best_val_loss = float('inf')
train_losses = []
val_losses = []

output_dir = "outputs/nepa-tiny-diffusion-cifar10"
os.makedirs(output_dir, exist_ok=True)

for epoch in range(TRAIN_CONFIG["num_epochs"]):
    model.train()
    epoch_loss = 0
    
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{TRAIN_CONFIG['num_epochs']}")
    
    for batch in progress_bar:
        pixel_values = batch["pixel_values"].to(device)
        
        # Forward pass
        outputs = model(pixel_values)
        loss = outputs.loss
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        
        # Update EMA
        update_ema(ema_decoder, model.decoder, ema_decay)
        
        epoch_loss += loss.item()
        global_step += 1
        
        progress_bar.set_postfix({"loss": f"{loss.item():.4f}", "lr": f"{scheduler.get_last_lr()[0]:.2e}"})
    
    avg_train_loss = epoch_loss / len(train_loader)
    train_losses.append(avg_train_loss)
    
    # Validation
    if (epoch + 1) % TRAIN_CONFIG["eval_interval"] == 0:
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for batch in val_loader:
                pixel_values = batch["pixel_values"].to(device)
                outputs = model(pixel_values)
                val_loss += outputs.loss.item()
        
        avg_val_loss = val_loss / len(val_loader)
        val_losses.append(avg_val_loss)
        
        print(f"Epoch {epoch+1}: Train Loss = {avg_train_loss:.4f}, Val Loss = {avg_val_loss:.4f}")
        
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            # Save best model
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'ema_decoder_state_dict': ema_decoder.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_loss': best_val_loss,
            }, f"{output_dir}/best_model.pt")
            print(f"  Best model saved!")
    
    # Save checkpoint
    if (epoch + 1) % TRAIN_CONFIG["save_interval"] == 0:
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'ema_decoder_state_dict': ema_decoder.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
        }, f"{output_dir}/checkpoint_epoch_{epoch+1}.pt")

print("Training complete!")

In [ ]:
# Plot training curves
plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.plot(train_losses)
plt.title("Training Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")

plt.subplot(1, 2, 2)
eval_epochs = list(range(TRAIN_CONFIG["eval_interval"]-1, len(train_losses), TRAIN_CONFIG["eval_interval"]))
plt.plot(eval_epochs, val_losses)
plt.title("Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")

plt.tight_layout()
plt.show()

## 5. Inference & Reconstruction

In [ ]:
# Load best model
checkpoint = torch.load(f"{output_dir}/best_model.pt")
model.load_state_dict(checkpoint['model_state_dict'])
ema_decoder.load_state_dict(checkpoint['ema_decoder_state_dict'])
model.eval()

print(f"Loaded best model from epoch {checkpoint['epoch']+1}")
print(f"Validation loss: {checkpoint['val_loss']:.4f}")

In [ ]:
# Reconstruct images
@torch.no_grad()
def reconstruct_images(model, images, num_inference_steps=50):
    """
    Reconstruct images from NEPA embeddings.
    
    Args:
        model: NepaConditionedDiffusionForReconstruction
        images: Input images (B, C, H, W) normalized to [-1, 1]
        num_inference_steps: Number of DDIM steps
    
    Returns:
        Reconstructed images (B, C, H, W) in [-1, 1]
    """
    model.eval()
    outputs = model.generate(
        pixel_values=images,
        num_inference_steps=num_inference_steps,
    )
    return outputs.images

# Get some test images
test_batch = next(iter(val_loader))
test_images = test_batch["pixel_values"][:8].to(device)

# Reconstruct
reconstructed = reconstruct_images(model, test_images, num_inference_steps=50)

print(f"Original shape: {test_images.shape}")
print(f"Reconstructed shape: {reconstructed.shape}")

## 6. Visualize Results

In [ ]:
def denormalize(tensor):
    """Convert from [-1, 1] to [0, 1] for display."""
    return (tensor + 1) / 2

def show_comparison(originals, reconstructed, num_images=8):
    """Show original vs reconstructed images side by side."""
    fig, axes = plt.subplots(2, num_images, figsize=(2*num_images, 4))
    
    for i in range(num_images):
        # Original
        orig = denormalize(originals[i]).cpu().permute(1, 2, 0).numpy()
        orig = np.clip(orig, 0, 1)
        axes[0, i].imshow(orig)
        axes[0, i].axis('off')
        if i == 0:
            axes[0, i].set_title('Original', fontsize=10)
        
        # Reconstructed
        recon = denormalize(reconstructed[i]).cpu().permute(1, 2, 0).numpy()
        recon = np.clip(recon, 0, 1)
        axes[1, i].imshow(recon)
        axes[1, i].axis('off')
        if i == 0:
            axes[1, i].set_title('Reconstructed', fontsize=10)
    
    plt.tight_layout()
    plt.show()

# Show comparison
show_comparison(test_images, reconstructed, num_images=8)

In [ ]:
# Compute reconstruction metrics
def compute_metrics(originals, reconstructed):
    """Compute MSE and PSNR between original and reconstructed images."""
    mse = F.mse_loss(reconstructed, originals).item()
    psnr = 10 * np.log10(4 / mse)  # 4 because range is [-1, 1] -> range 2, squared = 4
    return {"mse": mse, "psnr": psnr}

metrics = compute_metrics(test_images, reconstructed)
print(f"Reconstruction Metrics:")
print(f"  MSE: {metrics['mse']:.4f}")
print(f"  PSNR: {metrics['psnr']:.2f} dB")

In [ ]:
# Visualize the denoising process
@torch.no_grad()
def visualize_denoising(model, image, num_steps=10):
    """Visualize the denoising process step by step."""
    model.eval()
    
    # Get NEPA embeddings
    encoder_hidden_states = model.get_nepa_embeddings(image.unsqueeze(0))
    
    # Start with noise
    latent = torch.randn(1, 3, 224, 224, device=image.device)
    
    # Set up scheduler
    scheduler = model.inference_scheduler
    scheduler.set_timesteps(num_steps, device=image.device)
    
    intermediates = [latent.clone()]
    
    for t in scheduler.timesteps:
        pred_noise = model.decoder(latent, t.unsqueeze(0), encoder_hidden_states)
        latent = scheduler.step(pred_noise, t, latent).prev_sample
        intermediates.append(latent.clone())
    
    return intermediates

# Visualize
sample_image = test_images[0]
intermediates = visualize_denoising(model, sample_image, num_steps=10)

fig, axes = plt.subplots(1, len(intermediates), figsize=(2*len(intermediates), 2))
for i, img in enumerate(intermediates):
    img_np = denormalize(img[0]).cpu().permute(1, 2, 0).numpy()
    img_np = np.clip(img_np, 0, 1)
    axes[i].imshow(img_np)
    axes[i].axis('off')
    if i == 0:
        axes[i].set_title('Noise')
    elif i == len(intermediates) - 1:
        axes[i].set_title('Final')
    else:
        axes[i].set_title(f'Step {i}')

plt.suptitle('Denoising Process')
plt.tight_layout()
plt.show()

## 7. Save/Load Checkpoints

In [ ]:
# Save to Google Drive
import shutil

drive_output_dir = "/content/drive/MyDrive/nepa-checkpoints/nepa-tiny-diffusion-cifar10"
os.makedirs(drive_output_dir, exist_ok=True)

# Copy checkpoint
shutil.copy(f"{output_dir}/best_model.pt", f"{drive_output_dir}/best_model.pt")

# Save config
DIFFUSION_CONFIG.save_pretrained(drive_output_dir)

print(f"Checkpoint saved to {drive_output_dir}")

In [ ]:
# Load checkpoint from Google Drive (in a new session)
# Uncomment and run after mounting drive

# drive_output_dir = "/content/drive/MyDrive/nepa-checkpoints/nepa-tiny-diffusion-cifar10"
# 
# # Load config
# config = DiffusionDecoderConfig.from_pretrained(drive_output_dir)
# 
# # Create model
# model = NepaConditionedDiffusionForReconstruction(config)
# model.load_nepa_encoder(NEPA_MODEL_PATH)
# 
# # Load weights
# checkpoint = torch.load(f"{drive_output_dir}/best_model.pt")
# model.load_state_dict(checkpoint['model_state_dict'])
# model = model.to(device)
# model.eval()
# 
# print("Model loaded from Google Drive!")

## Notes

- **Prerequisites**: This notebook requires a pretrained NEPA encoder. Train it first using `nepa_cifar10_colab.ipynb`.
- **Training time**: ~2-3 hours for 100 epochs on a T4 GPU with CIFAR-10.
- **Memory**: The model uses ~4GB GPU memory with batch size 32.
- **Quality**: Reconstruction quality improves with more training epochs and inference steps.
- **Next steps**:
  - Try prefix completion (autoregressive generation)
  - Experiment with different UNet architectures
  - Compare with standard ViT features as conditioning